In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
df = pd.read_csv('analysed_data.csv')
df.head()

,area,room,living_room,age,floor,city,district,neighbourhood,price
0,90,2,1,6,6,istanbul,kagithane,caglayan,39500
1,75,2,1,29,2,istanbul,kagithane,ortabayir,28000
2,95,2,1,18,2,istanbul,maltepe,aydinevler,46000
3,65,2,1,9,0,istanbul,kucukcekmece,kemalpasa,22000
4,145,3,1,40,2,istanbul,sisli,mesrutiyet,95000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6059 entries, 0 to 6058
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   area           6059 non-null   int64 
 1   room           6059 non-null   int64 
 2   living_room    6059 non-null   int64 
 3   age            6059 non-null   int64 
 4   floor          6059 non-null   int64 
 5   city           6059 non-null   object
 6   district       6059 non-null   object
 7   neighbourhood  6059 non-null   object
 8   price          6059 non-null   int64 
dtypes: int64(6), object(3)
memory usage: 426.2+ KB


In [5]:
categorical_features = ['district', 'neighbourhood', 'city']
numerical_features = ['area', 'room', 'living_room', 'age', 'floor']

In [6]:
full_pipeline = ColumnTransformer([
    ('num', StandardScaler(), numerical_features), # Standart Scaler used to scale numerical features
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # One Hot Encoder used to encode categorical features
])

In [7]:
X = df.drop(columns=['price'], axis=1)
y = df['price']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=31)


In [9]:
model = Pipeline([
    ('preparation', full_pipeline),
    ('model', XGBRegressor(
        n_estimators=100,   # Tree count
        learning_rate=0.1,  # Learning rate
        max_depth=5,        # Tree depth
        random_state=31     
    ))
])

In [10]:
model.fit(X_train,y_train)

Pipeline(steps=[('preparation',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['area', 'room',
                                                   'living_room', 'age',
                                                   'floor']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['district', 'neighbourhood',
                                                   'city'])])),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=100, n_jobs=None,
                              num_parallel_tree=None, random_state=31, ...))])

In [11]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("R^2 Score:", r2)

Mean Squared Error: 141813168.0
Root Mean Squared Error: 11908.533410962074
R^2 Score: 0.662278950214386


In [15]:
example_data = {
    'area': [120],
    'room': [2],
    'living_room': [1],
    'age': [10],
    'floor': [2],
    'district': ['sultangazi'],
    'neighbourhood': ['gazi'],
    'city': ['istanbul']
}
print(model.predict(pd.DataFrame(example_data)))

[40688.59]
